# Visual Internal Reasoning: run the full pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chasemetoyer/visual-internal-reasoning/blob/main/colab_pipeline.ipynb)

This notebook runs the pipeline from the [README](https://github.com/chasemetoyer/visual-internal-reasoning#getting-started), one step per section:

1. **Setup**: GPU, Google Drive, code, VQGAN weights, unit test
2. **Generate** the synthetic dataset (train/val/test split by configuration)
3. **Audit** the dataset against its invariants
4. **Preprocess**: tokenize prompts and VQGAN-encode images
5. **Train** the image, text and filler models with identical settings
6. **Evaluate** all three models on the same held-out test items
7. **Look** at an imagined picture (optional)

Run the cells from top to bottom. Each step saves its output to Google Drive and skips itself when that output already exists, so after a disconnect you can run everything again from the top and finished steps are not repeated. A training run that gets interrupted starts again from step 0: training cannot resume mid-run.

## Before you start

- **GPU:** choose *Runtime → Change runtime type → GPU* (a T4 is enough). The setup cell stops if there is no GPU.
- **Google Drive:** a full run needs about 2.5 GB free: VQGAN weights (1 GB, downloaded once), preprocessed data (0.6 GB) and three final checkpoints (0.25 GB each).
- **Time:** a full run (100k samples, 30,000 training steps per model) takes several hours, mostly training and evaluation.
  - Leave `QUICK_TEST = True` for your first run. It runs the whole pipeline at toy size in about 15 minutes, which checks your setup.
  - Its training cells print `Done in ...s` for 200 steps. Multiply that by 150 to estimate one full 30,000-step run on your GPU.
- **Old checkpoints:** `imaginer_*.pth` files without `v2` in the name were trained with a model bug, and every script refuses them.

## 0. Settings

Edit these before running. Set `QUICK_TEST = False` for the real experiment; keep the other values unless you are deliberately changing the setup, so results stay comparable across the team.

In [ ]:
# --- SETTINGS ---
QUICK_TEST = True             # True: toy-size run that checks the setup. False: the full experiment.

DATASET_SIZE = 100_000        # samples, split 90/5/5 into train/val/test by configuration
SEED = 0                      # data seed: decides which configurations are held out for testing
TRAIN_SEED = 0                # weight init and batch order; the same for all three models
MAX_ITERS = 30_000            # training steps per model
VAL_INTERVAL = 1_000          # log validation loss every N steps
N_EVAL = 1_000                # distinct test items to score (at most ~1,040 at full size)
EVAL_SEED = 42                # which test items are picked, and the sampled imagination

REPO_URL = "https://github.com/chasemetoyer/visual-internal-reasoning"
REPO_REF = "main"             # branch, tag or commit to run
USE_DRIVE = True              # keep results in Google Drive (strongly recommended on Colab)
DRIVE_FOLDER = "visual-internal-reasoning"  # folder in My Drive for all runs

if QUICK_TEST:
    DATASET_SIZE, MAX_ITERS, VAL_INTERVAL, N_EVAL = 2_000, 200, 100, 48

## 1. Setup

### 1a. Paths, GPU and Google Drive

Results go to `My Drive/visual-internal-reasoning/seed<SEED>_size<DATASET_SIZE>/`. The 100k images and intermediate checkpoints stay on the Colab machine's fast local disk; only what later steps need is kept in Drive. Running from a local clone instead of Colab also works: results then go to `~/visual-internal-reasoning-runs/`.

In [ ]:
import codecs, os, shlex, shutil, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
PY = shlex.quote(sys.executable)                               # run every step with this notebook's Python
DATA_NAME = f"seed{SEED}_size{DATASET_SIZE}"                  # one dataset per data seed and size
TRAIN_NAME = f"iters{MAX_ITERS}_trainseed{TRAIN_SEED}"        # one set of models per training setup
if IN_COLAB:
    REPO_DIR = "/content/visual-internal-reasoning"
    SAVE_ROOT = f"/content/drive/MyDrive/{DRIVE_FOLDER}" if USE_DRIVE else "/content/results"
    LOCAL_ROOT = "/content/work"                               # fast, but erased when the session ends
else:
    REPO_DIR = os.getcwd()                                     # open the notebook from the repository root
    SAVE_ROOT = LOCAL_ROOT = os.path.expanduser("~/visual-internal-reasoning-runs")

VQGAN_CACHE = f"{SAVE_ROOT}/vqgan"                             # downloaded once, shared by every run
SAVE_DIR = f"{SAVE_ROOT}/{DATA_NAME}"
DATA_DIR = f"{LOCAL_ROOT}/{DATA_NAME}/dataset_imagination_v2"  # images: only needed until preprocessing
PROCESSED_DIR = f"{SAVE_DIR}/processed_v2"                     # train.pt, val.pt, test.pt
CKPT_DIR = f"{LOCAL_ROOT}/{DATA_NAME}/{TRAIN_NAME}/checkpoints"  # every checkpoint, including intermediate ones
FINAL_CKPT_DIR = f"{SAVE_DIR}/{TRAIN_NAME}/checkpoints"         # final checkpoints, kept
DATA_LOG_DIR = f"{SAVE_DIR}/logs"                               # output of generate, audit, preprocess
LOG_DIR = f"{SAVE_DIR}/{TRAIN_NAME}/logs"                       # output of train, evaluate, visualize

if IN_COLAB:
    gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                         shell=True, capture_output=True, text=True)
    if gpu.returncode != 0:
        raise RuntimeError("No GPU found. Choose Runtime > Change runtime type > GPU, then run this cell again.")
    print("GPU:", gpu.stdout.strip())
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
else:
    print("Not on Colab: the scripts use CUDA, then Apple MPS, then the CPU, whichever is available.")

for d in (VQGAN_CACHE, PROCESSED_DIR, FINAL_CKPT_DIR, DATA_LOG_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)

def run(cmd, log=None, cwd=None):
    """Runs a shell command (from the repository root unless cwd is given), streams its output here
    and into the log file, and raises if it fails so the notebook stops at the broken step."""
    print(f"$ {cmd}\n", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO_DIR, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    with open(log or os.devnull, "wb") as log_file:
        while chunk := proc.stdout.read1(8192):
            sys.stdout.write(decoder.decode(chunk))
            sys.stdout.flush()
            log_file.write(chunk)
    if proc.wait() != 0:
        raise RuntimeError(f"Step failed with exit code {proc.returncode}; the output above shows why.")

def train(fmt):
    """Trains one format with the shared settings and keeps its final checkpoint in FINAL_CKPT_DIR."""
    final = f"{FINAL_CKPT_DIR}/imaginer_v2_{fmt}_final.pth"
    if os.path.exists(final):
        print(f"Already trained: {final}\nDelete that file to train this model again.")
        return
    run(f'{PY} src/train_imaginer.py --format {fmt} --data "{PROCESSED_DIR}" --out-dir "{CKPT_DIR}" '
        f'--max-iters {MAX_ITERS} --val-interval {VAL_INTERVAL} --seed {TRAIN_SEED}', log=f"{LOG_DIR}/4_train_{fmt}.log")
    produced = f"{CKPT_DIR}/imaginer_v2_{fmt}_final.pth"
    if os.path.abspath(produced) != os.path.abspath(final):
        shutil.copy(produced, final)
    print("Kept", final)

print("Results folder:", SAVE_DIR)

### 1b. Get the code

On Colab this clones the repository and installs its requirements; the preinstalled PyTorch is kept. Running locally uses the checkout the notebook was opened from and installs nothing.

In [ ]:
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        run(f"git clone -q {REPO_URL} {REPO_DIR} && git -C {REPO_DIR} checkout -q {REPO_REF}", cwd="/content")
    run(f"{PY} -m pip install -q -r requirements.txt")
else:
    assert os.path.exists(f"{REPO_DIR}/src/datafactory.py"), "Open this notebook from the repository root."
run("git log --oneline -1")

### 1c. VQGAN weights

The pretrained VQGAN (ImageNet, f16, 16,384 codes) from [Taming Transformers](https://github.com/CompVis/taming-transformers), about 1 GB. It is downloaded into Drive once and linked into the repository. A checkpoint of only a few KB means the download failed: delete it from `vqgan/` in the results folder and run this cell again.

In [ ]:
VQGAN_FILES = {  # name: (download URL, minimum plausible size in bytes)
    "vqgan_imagenet_f16_16384.yaml": ("https://heibox.uni-heidelberg.de/d/a7530b09fed84f80a887/files/?p=%2Fconfigs%2Fmodel.yaml&dl=1", 500),
    "vqgan_imagenet_f16_16384.ckpt": ("https://heibox.uni-heidelberg.de/d/a7530b09fed84f80a887/files/?p=%2Fckpts%2Flast.ckpt&dl=1", 900_000_000),
}
for name, (url, min_size) in VQGAN_FILES.items():
    target, cached = f"{REPO_DIR}/{name}", f"{VQGAN_CACHE}/{name}"
    if os.path.exists(target) and os.path.getsize(target) >= min_size:
        print(f"{name}: ready ({os.path.getsize(target):,} bytes)")
        continue
    if not (os.path.exists(cached) and os.path.getsize(cached) >= min_size):
        run(f'curl -L --fail -o "{cached}" "{url}"')
        if os.path.getsize(cached) < min_size:
            raise RuntimeError(f"{cached} is only {os.path.getsize(cached)} bytes: the download failed. Delete it and retry.")
    if os.path.lexists(target):
        os.remove(target)
    os.symlink(cached, target)
    print(f"{name}: linked from {cached}")

### 1d. Unit test

Checks that each transformer block adds its attention output to the residual stream (the bug that invalidated the old results).

In [ ]:
run(f"{PY} tests/test_block.py")

## 2. Generate the dataset

Draws both shapes exactly at grid-cell centres, labels each sample from the drawing, and splits unique configurations (square start, circle position, move) 90/5/5 into train/val/test with `SEED`, so no test prompt is ever trained on. The 100k images take a few minutes.

In [ ]:
processed = all(os.path.exists(f"{PROCESSED_DIR}/{s}.pt") for s in ("train", "val", "test"))
if processed:
    print("Already preprocessed in an earlier session, so the images are not needed again.")
elif os.path.exists(f"{DATA_DIR}/manifest.json"):
    print("Already generated:", DATA_DIR)
else:
    if os.path.isdir(DATA_DIR):
        shutil.rmtree(DATA_DIR)  # an earlier generation into this folder was interrupted
    run(f'{PY} src/datafactory.py --size {DATASET_SIZE} --seed {SEED} --out "{DATA_DIR}"', log=f"{DATA_LOG_DIR}/1_datafactory.log")

## 3. Audit the dataset

Re-checks every invariant from the written files alone (prompts and pixels) and prints a failure rate for each. Every row should read 0.00%, the Yes share 50%, and the near-miss share of No about 50%. The last row says how many test pictures also appear in training under a different prompt: expected, since the split is by configuration.

In [ ]:
if os.path.isdir(DATA_DIR):
    run(f'{PY} src/audit_datafactory.py "{DATA_DIR}"', log=f"{DATA_LOG_DIR}/2_audit.log")
else:
    print("The images were removed with the last session; the audit from that session is in", f"{DATA_LOG_DIR}/2_audit.log")

## 4. Preprocess

Tokenizes prompts and encodes each image into 256 VQGAN tokens, writing `train.pt`, `val.pt` and `test.pt` (with all three sequence formats) to Drive. At full size this is the slowest step before training, on the order of an hour.

In [ ]:
if all(os.path.exists(f"{PROCESSED_DIR}/{s}.pt") for s in ("train", "val", "test")):
    print("Already preprocessed:", PROCESSED_DIR)
else:
    run(f'{PY} src/preprocess.py --data "{DATA_DIR}" --out "{PROCESSED_DIR}"', log=f"{DATA_LOG_DIR}/3_preprocess.log")

## 5. Train

Three models with identical settings and one shared vocabulary; only the sequence format differs:

| Format | Sequence | Loss on |
| --- | --- | --- |
| `image` | prompt, 256 image tokens, answer | image tokens and answer |
| `text` | prompt, answer | answer |
| `filler` | prompt, 256 `<THINK>` tokens, answer | answer |

Each run first checks its loss mask, then prints the training loss every 100 steps, and the validation loss every `VAL_INTERVAL` steps and at the end. Use the **val answer loss** to compare the text and filler models only. The image model predicts its validation answer with the true picture already in its input (teacher forcing), so its answer loss is near zero whether or not it can imagine the picture; compare it through the *imagined* rows of the evaluation in section 6. Its overall val loss is mostly the 256 image tokens.

The three cells are independent, so you can run them in separate sessions. Each final checkpoint is kept in Drive.

In [ ]:
train("image")

In [ ]:
train("text")

In [ ]:
train("filler")

## 6. Evaluate

Scores `N_EVAL` held-out test items with distinct configurations (half Yes, a quarter near-miss No, a quarter far No), runs every trained model on the same items, and prints each accuracy with a 95% confidence interval and the majority-class baseline. It refuses a checkpoint that trained on any of these test configurations. Models not trained yet are left out. At full size this takes about an hour, mostly the image model generating its 256-token pictures.

Read the results from the confidence intervals: overlapping intervals mean the difference is not established. The `VERDICT` lines at the end use fixed margins rather than the intervals.

In [ ]:
ckpts = {fmt: f"{FINAL_CKPT_DIR}/imaginer_v2_{fmt}_final.pth" for fmt in ("image", "text", "filler")}
trained = {fmt: path for fmt, path in ckpts.items() if os.path.exists(path)}
if len(trained) < len(ckpts):
    print("Not trained yet, so left out:", ", ".join(sorted(set(ckpts) - set(trained))))
flags = " ".join(f'--{fmt}-ckpt "{path}"' for fmt, path in trained.items())
run(f'{PY} evaluation/evaluate_rigorous.py --data "{PROCESSED_DIR}" --n {N_EVAL} --seed {EVAL_SEED} {flags}',
    log=f"{LOG_DIR}/5_evaluate.log")
print("\nSaved to", f"{LOG_DIR}/5_evaluate.log")

## 7. Look at an imagined picture (optional)

Lets the image model imagine the picture for the demo prompt in `visualization/visualize.py` and decodes it with the VQGAN. That prompt moves the square "0 down", a move the training data never contains, so treat the picture as an illustration only.

In [ ]:
from IPython.display import Image, display

image_ckpt = f"{FINAL_CKPT_DIR}/imaginer_v2_image_final.pth"
if not os.path.exists(image_ckpt):
    print("Train the image model first (section 5).")
else:
    vis_dir = f"{LOCAL_ROOT}/{DATA_NAME}/{TRAIN_NAME}/visualize"
    os.makedirs(vis_dir, exist_ok=True)
    # visualize.py reads the VQGAN files from, and writes final_dream.png to, its working directory
    for name in ("vqgan_imagenet_f16_16384.yaml", "vqgan_imagenet_f16_16384.ckpt", "taming"):
        if os.path.exists(f"{REPO_DIR}/{name}") and not os.path.lexists(f"{vis_dir}/{name}"):
            os.symlink(f"{REPO_DIR}/{name}", f"{vis_dir}/{name}")
    run(f'{PY} "{REPO_DIR}/visualization/visualize.py" --ckpt "{image_ckpt}"',
        log=f"{LOG_DIR}/6_visualize.log", cwd=vis_dir)
    shutil.copy(f"{vis_dir}/final_dream.png", f"{SAVE_DIR}/{TRAIN_NAME}/final_dream.png")
    display(Image(f"{vis_dir}/final_dream.png"))

## Where everything is

In the results folder (`My Drive/visual-internal-reasoning/seed<SEED>_size<DATASET_SIZE>/`):

- `processed_v2/`: the preprocessed splits, reused by any later training run on this dataset
- `iters<MAX_ITERS>_trainseed<TRAIN_SEED>/checkpoints/`: the three final checkpoints
- `logs/`: the full output of generating, auditing and preprocessing the dataset
- `iters<MAX_ITERS>_trainseed<TRAIN_SEED>/logs/`: the full output of training, evaluation and visualization; the results table is in `5_evaluate.log`
- `iters<MAX_ITERS>_trainseed<TRAIN_SEED>/final_dream.png`: the last imagined picture from section 7

**Troubleshooting**

- *Disconnected?* Run all cells again from the top. Finished steps skip themselves; an interrupted training run starts over.
- *Redo a step?* Delete its output (the file or folder its skip message names) and run its cell again.
- *A step failed?* The cell stops with the error; the full output is also in that step's log file.
- *Another data seed or dataset size?* Change `SEED` or `DATASET_SIZE`; the run gets its own folder.